# Librerias

In [7]:
import re
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from typing import List, Tuple, Dict, Iterable, Optional, Set
from copy import deepcopy

In [ ]:
CSV_PATH = r"D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tareaunidad2\train.csv"
# Binning (discretización) para numéricos
AGE_BINS = [0, 2, 12, 18, 30, 45, 60, 80] 
FARE_BINS = [0, 7.91, 14.45, 31, 120, 600]

# Control de qué atributos tokenizar
USE_TOKENS = dict(
    survival=True,
    pclass=True,
    sex=True,
    title=True,        
    age=True,
    fare=True,
    embarked=True,
    familysize=True,   
    isalone=True      
)

EXCLUDE_MISSING = True
# ---- FP-Growth ----
MINSUP_ITEMSETS_REL = 0.08   # 8% de pasajeros 
MINSUP_ITEMSETS_ABS = None 
MIN_ITEMSET_SIZE    = 2      # mínimo tamaño del itemset 

# ---- PrefixSpan ----
SEQUENCE_GROUP_BY = 'ticket'    
MINSUP_SEQUENCES_REL = 0.02    
MINSUP_SEQUENCES_ABS = None
MIN_PATTERN_LENGTH = 1          
MIN_GROUP_SIZE = 2

TARGETS = ["survival=1"]   
TOP_RULES_SHOW = 20


In [12]:
df_raw = pd.read_csv(CSV_PATH)

# Normalizar nombres estándar Kaggle si vienen en minúsculas o alternativos
rename_map = {
    'survival':'Survived','pclass':'Pclass','sex':'Sex','age':'Age','sibsp':'SibSp',
    'parch':'Parch','ticket':'Ticket','fare':'Fare','cabin':'Cabin','embarked':'Embarked'
}
for k, v in rename_map.items():
    if k in df_raw.columns and v not in df_raw.columns:
        df_raw.rename(columns={k: v}, inplace=True)

# Extraer Title desde Name
def extract_title(name: str) -> str:
    if pd.isna(name): return "Unknown"
    m = re.search(r",\s*([^\.]+)\.", str(name))
    return m.group(1).strip() if m else "Unknown"

df_aug = df_raw.copy()
if "Name" in df_aug.columns:
    df_aug["Title"] = df_aug["Name"].apply(extract_title)
else:
    df_aug["Title"] = "Unknown"

# Agrupar títulos raros
title_map = {
    "Mlle":"Miss", "Ms":"Miss", "Mme":"Mrs",
    "Lady":"Noble", "Countess":"Noble", "Sir":"Noble", "Don":"Noble", "Dona":"Noble",
    "Jonkheer":"Noble", "Dr":"Dr", "Rev":"Rev", "Major":"Officer", "Col":"Officer", "Capt":"Officer"
}
df_aug["Title"] = df_aug["Title"].replace(title_map)

# FamilySize e IsAlone
df_aug["FamilySize"] = df_aug["SibSp"].fillna(0).astype(int) + df_aug["Parch"].fillna(0).astype(int) + 1
df_aug["IsAlone"]    = (df_aug["FamilySize"] == 1).astype(int)

df_aug.head(5)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,FamilySize,IsAlone
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr,2,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,2,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss,1,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs,2,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Mr,1,1


# Funciones auxiliares

In [ ]:
# Tokenización a canastas de tokens y construcción de secuencias 
def cut_one(x, bins, labels=None):
    if pd.isna(x): return "missing"
    return pd.cut([x], bins=bins, labels=labels, include_lowest=True)[0]

def tokens_row(r: pd.Series) -> set:
    t = set()
    # survival
    if USE_TOKENS.get("survival", False) and "Survived" in r:
        t.add(f"survival={int(r['Survived'])}" if pd.notna(r["Survived"]) else "survival=missing")
    # pclass
    if USE_TOKENS.get("pclass", False) and "Pclass" in r:
        t.add(f"pclass={int(r['Pclass'])}" if pd.notna(r["Pclass"]) else "pclass=missing")
    # sex
    if USE_TOKENS.get("sex", False) and "Sex" in r:
        t.add(f"sex={str(r['Sex']).strip().lower() if pd.notna(r['Sex']) else 'missing'}")
    # title
    if USE_TOKENS.get("title", False) and "Title" in r:
        t.add(f"title={str(r['Title']) if pd.notna(r['Title']) else 'Unknown'}")
    # age
    if USE_TOKENS.get("age", False) and "Age" in r:
        t.add(f"age_bin={cut_one(r['Age'], AGE_BINS)}")
    # fare
    if USE_TOKENS.get("fare", False) and "Fare" in r:
        t.add(f"fare_bin={cut_one(r['Fare'], FARE_BINS)}")
    # embarked
    if USE_TOKENS.get("embarked", False) and "Embarked" in r:
        v = str(r["Embarked"]).upper() if pd.notna(r["Embarked"]) else "missing"
        t.add(f"embarked={v}")
    # FamilySize
    if USE_TOKENS.get("familysize", False) and "FamilySize" in r:
        fs = int(r["FamilySize"]) if pd.notna(r["FamilySize"]) else -1
        fs_bin = "1" if fs==1 else ("2-3" if 2<=fs<=3 else ("4plus" if fs>=4 else "missing"))
        t.add(f"familysize_bin={fs_bin}")
    # IsAlone
    if USE_TOKENS.get("isalone", False) and "IsAlone" in r:
        t.add(f"isalone={int(r['IsAlone']) if pd.notna(r['IsAlone']) else -1}")

    if EXCLUDE_MISSING:
        t = {x for x in t if not x.endswith("=missing")}
    return t

transactions = df_aug.apply(tokens_row, axis=1).tolist()
baskets = [b for b in transactions if b]
print(f"[INFO] Baskets (pasajeros): {len(baskets)} | Ejemplo tokens:", list(next(iter(baskets)))[:8])

# Secuencias (grupos por ticket o apellido)
if SEQUENCE_GROUP_BY == 'ticket' and 'Ticket' in df_aug.columns:
    group_key = df_aug['Ticket'].fillna('missing')
elif SEQUENCE_GROUP_BY == 'surname' and 'Name' in df_aug.columns:
    # Apellido desde Name 
    def surname_from_name(name: str) -> str:
        if pd.isna(name): return "missing"
        m = re.match(r"\s*([^,]+),", str(name))
        return m.group(1).strip() if m else str(name).strip()
    group_key = df_aug['Name'].apply(surname_from_name).fillna('missing')
else:
    group_key = pd.Series(['ALL']*len(df_aug))

df_tmp = pd.DataFrame({
    'group': group_key,
    'order': df_aug.get('PassengerId', pd.Series(range(1, len(df_aug)+1))),
    'tokens': transactions
}).sort_values(['group','order']).reset_index(drop=True)

sequences: List[List[Set[str]]] = []
for g, sub in df_tmp.groupby('group'):
    if len(sub) >= MIN_GROUP_SIZE:
        seq = [set(t) for t in sub['tokens']]
        seq = [e for e in seq if e]
        if seq:
            sequences.append(seq)

print(f"[INFO] Secuencias (grupo={SEQUENCE_GROUP_BY}, size≥{MIN_GROUP_SIZE}): {len(sequences)}")


[INFO] Baskets (pasajeros): 891 | Ejemplo tokens: ['isalone=0', 'sex=male', 'embarked=S', 'title=Mr', 'pclass=3', 'survival=0', 'age_bin=(18.0, 30.0]', 'fare_bin=(-0.001, 7.91]']
[INFO] Secuencias (grupo=ticket, size≥2): 134


In [14]:
def resolve_min_support_rel_abs(total: int, rel: Optional[float], abs_sup: Optional[int]) -> int:
    if abs_sup is not None:
        return max(1, int(abs_sup))
    if rel is not None:
        return max(1, int(rel * total + 1e-9))
    raise ValueError("Define soporte relativo o absoluto.")

MIN_SUP_ITEMSETS  = resolve_min_support_rel_abs(len(baskets),   MINSUP_ITEMSETS_REL,  MINSUP_ITEMSETS_ABS)
MIN_SUP_SEQUENCES = resolve_min_support_rel_abs(len(sequences), MINSUP_SEQUENCES_REL, MINSUP_SEQUENCES_ABS)
print(f"[INFO] min_sup itemsets (abs):   {MIN_SUP_ITEMSETS}")
print(f"[INFO] min_sup secuencias (abs): {MIN_SUP_SEQUENCES}")


[INFO] min_sup itemsets (abs):   71
[INFO] min_sup secuencias (abs): 2


# FP-Growth

In [ ]:
def _build_flist(transactions: List[List[str]], min_support: int):
    freq = Counter()
    for t in transactions:
        freq.update(t)
    freq = Counter({i: c for i, c in freq.items() if c >= min_support})
    order_list = [i for i, _ in sorted(freq.items(), key=lambda kv: (-kv[1], kv[0]))]
    return order_list, freq

def _insert_transaction(tree_root: dict, header: Dict[str, Dict], ordered_items: List[str]):
    node = tree_root
    for itm in ordered_items:
        if itm not in node['children']:
            child = {'item': itm, 'count': 0, 'parent': node, 'children': {}, 'link': None}
            node['children'][itm] = child
            if header[itm]['head'] is None:
                header[itm]['head'] = child
            else:
                cur = header[itm]['head']
                while cur['link'] is not None:
                    cur = cur['link']
                cur['link'] = child
        node = node['children'][itm]
        node['count'] += 1

def _build_fptree(transactions: List[List[str]], min_support: int):
    order_list, freq_counter = _build_flist(transactions, min_support)
    if not order_list:
        return None, None, None
    header = {i: {'support': freq_counter[i], 'head': None} for i in order_list}
    root = {'item': None, 'count': 0, 'parent': None, 'children': {}, 'link': None}
    for t in transactions:
        ordered = [i for i in order_list if i in t]
        if ordered:
            _insert_transaction(root, header, ordered)
    return root, header, order_list

def _is_single_path(root: dict) -> bool:
    node = root
    while True:
        if len(node['children']) > 1:
            return False
        if len(node['children']) == 0:
            return True
        node = next(iter(node['children'].values()))

def _conditional_pattern_base(header: Dict[str, Dict], item: str) -> List[List[str]]:
    patterns = []
    node = header[item]['head']
    while node is not None:
        path = []
        parent = node['parent']
        while parent and parent['item'] is not None:
            path.append(parent['item'])
            parent = parent['parent']
        if path:
            for _ in range(node['count']):
                patterns.append(path)
        node = node['link']
    return patterns

def fpgrowth(transactions: List[Iterable[str]],
             min_support_abs: int,
             min_itemset_size: int = 1):
    T = [list(set(t)) for t in transactions if t]
    if not T:
        return []
    root, header, _ = _build_fptree(T, min_support_abs)
    if root is None:
        return []

    results = []

    def mine(tree_root: dict, header_tbl: Dict[str, Dict], suffix: Tuple[str, ...]):
        if _is_single_path(tree_root):
            path_items = []
            node = tree_root
            while node['children']:
                node = next(iter(node['children'].values()))
                path_items.append((node['item'], node['count']))
            from itertools import combinations
            for r in range(1, len(path_items)+1):
                for combo in combinations(path_items, r):
                    items = tuple(sorted([i for i, _ in combo]))
                    sup = min(c for _, c in combo)
                    full = tuple(sorted(items + suffix))
                    if len(full) >= min_itemset_size:
                        results.append((full, sup))
            return

        items_sorted = sorted(header_tbl.items(), key=lambda kv: (kv[1]['support'], kv[0]))
        for item, meta in items_sorted:
            new_suffix = tuple(sorted((item,) + suffix))
            if len(new_suffix) >= min_itemset_size:
                results.append((new_suffix, meta['support']))

            cond_patterns = _conditional_pattern_base(header_tbl, item)
            cond_root, cond_header, _ = _build_fptree(cond_patterns, min_support_abs)
            if cond_root and cond_header:
                mine(cond_root, cond_header, new_suffix)

    mine(root, header, ())
    # deduplicar (máximo soporte)
    best = {}
    for items, sup in results:
        best[tuple(sorted(items))] = max(best.get(tuple(sorted(items)), 0), sup)
    out = [(k, v) for k, v in best.items()]
    out.sort(key=lambda x: (-x[1], x[0]))
    return out


itemsets = fpgrowth([list(b) for b in baskets],
                    min_support_abs=MIN_SUP_ITEMSETS,
                    min_itemset_size=MIN_ITEMSET_SIZE)

df_fpg = pd.DataFrame(
    [{"itemset": " | ".join(items), "size": len(items), "support": sup}
     for items, sup in itemsets]
).sort_values(["size","support"], ascending=[True, False]).reset_index(drop=True)

print(f"[INFO] Itemsets encontrados: {len(df_fpg)}")
df_fpg.head(20)


[INFO] Itemsets encontrados: 811


,itemset,size,support
0,familysize_bin=1 | isalone=1,2,537
1,sex=male | title=Mr,2,517
2,sex=male | survival=0,2,468
3,embarked=S | sex=male,2,441
4,survival=0 | title=Mr,2,436
5,embarked=S | survival=0,2,427
6,familysize_bin=1 | sex=male,2,411
7,isalone=1 | sex=male,2,411
8,embarked=S | title=Mr,2,397
9,familysize_bin=1 | title=Mr,2,397
